# CNNs avec Keras sur des données de paysage

Dans ces travaux pratiques, nous classons les photos du jeu des paysages (Intel Image Classification, 6 classes) :

1. avec le CNN des slides, puis un CNN plus profond, que vous écrivez (facultatif : avec normalisation par batch, AdamW et warmup) ;
2. en augmentant les données par des couches ;
3. avec trois architectures historiques : LeNet, AlexNet et un bloc Inception ;
4. en analysant les erreurs sur l'ensemble de test ;
5. par transfert d'apprentissage : EfficientNetV2B0 pré-entraîné sur ImageNet, gelé puis fine-tuné (partie « Transfert d'apprentissage ») ;
6. facultatif : avec un *Vision Transformer* pré-entraîné (partie « Transfert d'apprentissage avec des transformeurs »).

La partie 5 va avec le chapitre « Transfer learning » des slides : arrêtez-vous avant elle si vous n'en êtes qu'au chapitre sur les CNN. La partie 6 est facultative : elle reprend la démarche de la partie 5 avec une architecture, le *Vision Transformer*, que la partie résume en quelques lignes et que les slides présentent plus tard (chapitre « Transformers »).

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'exécution` et vérifiez que l'accélérateur matériel est un GPU : sans lui, chaque entraînement de ce TP prendrait des heures.

In [ ]:
!nvidia-smi

## Téléchargement du jeu des paysages

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Imports

In [ ]:
!pip install -q keras-hub
import itertools
import pathlib

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.metrics
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Préparation des données

Comme dans les slides, `keras.utils.image_dataset_from_directory` lit les images de `seg_train` par batchs de 32, dossier par dossier : l'étiquette d'une image est l'indice de son dossier, dans l'ordre alphabétique (0 = `buildings`, …, 5 = `street`). 20 % des images servent à la validation, avec un découpage fixé par `seed=0`.

Les images restent sur le disque et sont lues à chaque époque. En mémoire, en flottants de 4 octets, les 14 034 images d'entraînement prendraient 3,8 Go en 150 × 150 et 8,5 Go en 224 × 224 (la taille du transfert d'apprentissage) : plus que ce que Colab laisse libre à côté du modèle, sur ses 12,7 Go de RAM.

L'ensemble de test (`seg_test`) est lu sans mélange (`shuffle=False`) : les prédictions du modèle sortent dans l'ordre de `dataset_test.file_paths`, celui de `etiquettes_test`.

In [ ]:
def charge_paysages(taille_image: tuple[int, int]) -> tuple:
  """Charge les images d'entraînement, de validation et de test à cette taille."""
  dataset_ent, dataset_val = keras.utils.image_dataset_from_directory(
    "landscape/seg_train",
    image_size=taille_image,
    validation_split=0.2,  # 20 % des images pour la validation
    subset="both",
    seed=0,  # le même découpage à chaque exécution
  )
  dataset_test = keras.utils.image_dataset_from_directory(
    "landscape/seg_test", image_size=taille_image, shuffle=False
  )
  return dataset_ent, dataset_val, dataset_test


dataset_ent, dataset_val, dataset_test = charge_paysages((150, 150))
noms_etiquettes = dataset_ent.class_names
# Chemins et étiquettes des images de test, dans l'ordre des prédictions
chemins_test = numpy.array(dataset_test.file_paths)
etiquettes_test = numpy.concatenate([etiquettes for _, etiquettes in dataset_test])
print(noms_etiquettes)

## Regardons les données

In [ ]:
# Un batch : 32 images 150 × 150 à 3 canaux, et leurs 32 étiquettes
images, etiquettes = next(iter(dataset_ent))
print(f"Forme d'un batch d'images : {images.shape}")
print(f"Forme d'un batch d'étiquettes : {etiquettes.shape}")

effectifs = [
  len(list((pathlib.Path("landscape/seg_train") / nom).iterdir()))
  for nom in noms_etiquettes
]
plt.bar(noms_etiquettes, effectifs)
plt.title("Nombre d'images par classe (entraînement et validation)")
plt.ylabel("Nombre d'images")
plt.xlabel("Classe")
plt.show()

In [ ]:
# Les pixels sont des flottants de 0 à 255 : imshow les attend en entiers
images = numpy.asarray(images).astype("uint8")
_, ax = plt.subplots(5, 5, figsize=(15, 15))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  ax[i, j].imshow(images[k])
  ax[i, j].set_title(noms_etiquettes[int(etiquettes[k])])
  ax[i, j].axis("off")
plt.show()

## Création du modèle

Le CNN des slides (« Définir un CNN ») : deux blocs convolution + pooling, puis un pooling global qui garde une valeur par canal, et une couche dense de sortie. `Rescaling(1 / 255)` ramène les pixels de 0–255 à 0–1, l'échelle où un réseau entraîné depuis zéro apprend bien.

In [ ]:
modele = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),  # pixels 0–255 → 0–1
    layers.Conv2D(32, 3, activation="relu"),  # 32 filtres 3 × 3
    layers.MaxPooling2D(2),  # fenêtres 2 × 2, pas de 2
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.GlobalAveragePooling2D(),  # une valeur par canal
    layers.Dense(6, activation="softmax"),  # une probabilité par classe
  ]
)
modele.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-3),
  loss="sparse_categorical_crossentropy",  # étiquettes entières
  metrics=["accuracy"],
)
modele.summary()

## Pouvez-vous expliquer les différents nombres de paramètres ?

Retrouvez par le calcul le nombre de paramètres de chaque couche du résumé. Pourquoi certaines couches n'en ont-elles aucun ?

### Solution

- `conv2d` : chacun des 32 filtres a 3 × 3 poids par canal d'entrée, sur 3 canaux (RVB), plus un biais : (3 × 3 × 3 + 1) × 32 = 896.
- `conv2d_1` : 64 filtres, qui couvrent chacun les 32 canaux de la couche précédente : (3 × 3 × 32 + 1) × 64 = 18 496.
- `dense` : 64 entrées (une par canal, après le pooling global) et 6 sorties, plus 6 biais : 64 × 6 + 6 = 390.
- `rescaling`, les poolings et le pooling global n'apprennent rien : ils appliquent une opération fixe (multiplier, prendre un maximum ou une moyenne).

Total : 19 782 paramètres. Le pooling global garde la tête petite : avec `Flatten`, la couche dense recevrait 36 × 36 × 64 = 82 944 valeurs, soit 497 670 paramètres.

## Apprentissage

Entraînons ce modèle comme dans les slides (« Entraîner avec arrêt précoce ») : au plus 10 époques, arrêtées après 3 époques sans baisse de la perte de validation, avec retour aux meilleurs poids.

Un callback peut garder son état d'un `fit` à l'autre : selon la version de Keras 3, un même `EarlyStopping` repart de la meilleure perte de validation de l'entraînement précédent, et il arrêterait alors le modèle suivant au bout de 3 époques en lui rendant les poids de sa première époque. `cree_arret_precoce()` en crée donc un neuf pour chaque entraînement.

`entraine(modele, nb_epoques, taux_apprentissage)` regroupe ensuite ces étapes (compilation, entraînement avec arrêt précoce, courbes) pour les modèles suivants.

In [ ]:
def cree_arret_precoce() -> keras.callbacks.EarlyStopping:
  """Un arrêt précoce neuf : 3 époques de patience, retour aux meilleurs poids."""
  return keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
  )


historique = modele.fit(
  dataset_ent, validation_data=dataset_val, epochs=10, callbacks=[cree_arret_precoce()]
)


def trace_mesures(historique: keras.callbacks.History) -> None:
  """Trace l'exactitude et la perte, en entraînement et en validation."""
  for mesure, nom in (("accuracy", "Exactitude"), ("loss", "Perte")):
    plt.plot(historique.history[mesure], label="Entraînement")
    plt.plot(historique.history[f"val_{mesure}"], label="Validation")
    plt.title(f"{nom} du modèle")
    plt.ylabel(nom)
    plt.xlabel("Époque")
    plt.legend()
    plt.show()


trace_mesures(historique)


def entraine(
  modele: keras.Model, nb_epoques: int, taux_apprentissage: float = 1e-4
) -> keras.callbacks.History:
  """Compile et entraîne un modèle sur les images 150 × 150, puis trace ses courbes."""
  modele.compile(
    optimizer=keras.optimizers.Adam(learning_rate=taux_apprentissage),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  historique = modele.fit(
    dataset_ent,
    validation_data=dataset_val,
    epochs=nb_epoques,
    callbacks=[cree_arret_precoce()],
  )
  trace_mesures(historique)
  return historique

## Améliorez cette performance

Le CNN des slides plafonne vers 66 % d'exactitude : deux convolutions ne voient que de petits motifs, trop peu pour reconnaître une scène. Écrivez une fonction `construit_cnn_profond()` qui renvoie un CNN plus profond, avec des poids neufs à chaque appel, puis entraînez-le avec `entraine` :

- des blocs convolution + pooling répétés, jusqu'à des cartes de quelques pixels de côté ;
- du *dropout* avant les couches denses de la tête, contre le surapprentissage ;
- entre 10 et 20 époques, chacune en moins d'une minute sur le GPU de Colab.

Objectif : plus de 70 % d'exactitude en validation. La solution atteint environ 86 % en 15 époques d'environ 45 secondes. `construit_cnn_profond()` resservira pour l'augmentation de données.

**Documentation utile** :

- https://keras.io/api/layers/reshaping_layers/flatten/
- https://keras.io/api/layers/regularization_layers/dropout/

In [ ]:
def construit_cnn_profond() -> keras.Sequential: ...


# Votre code ici

### Conseils supplémentaires

- Entrée : `keras.Input(shape=(150, 150, 3))` puis `layers.Rescaling(1 / 255)`, comme le CNN des slides.
- Un bloc : `layers.Conv2D(200, 3, activation="relu", padding="same", kernel_initializer="orthogonal")` puis `layers.MaxPooling2D(2, padding="same")`. Avec `padding="same"`, la convolution garde la taille des cartes et seul le pooling la divise par 2, arrondie au-dessus : 150 → 75 → 38 → 19 → 10 → 5 → 3 en 6 blocs. Une 7ᵉ convolution, sans pooling, termine sur des cartes 3 × 3.
- Tête : `layers.Flatten()`, puis trois couches `Dense` ReLU de 200, 100 et 50 neurones, et la sortie `layers.Dense(6, activation="softmax")`, chacune précédée de `layers.Dropout(0.2)`.
- Entraînement : `entraine(modele, nb_epoques=15)`, avec le taux d'apprentissage par défaut de `entraine` (1e-4).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_cnn_profond() -> keras.Sequential:
  blocs = []
  for _ in range(6):
    blocs += [layers.Conv2D(...), layers.MaxPooling2D(...)]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocs,
      layers.Conv2D(...),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      ...,
      layers.Dense(6, activation="softmax"),
    ]
  )


modele = construit_cnn_profond()
modele.summary()
historique = entraine(modele, nb_epoques=...)
```

### Solution

In [ ]:
def construit_cnn_profond() -> keras.Sequential:
  """Le CNN profond de la solution, avec des poids neufs à chaque appel."""
  blocs = []
  for _ in range(6):  # cartes de 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels de côté
    blocs += [
      layers.Conv2D(
        200,
        3,
        activation="relu",
        padding="same",
        kernel_initializer="orthogonal",
      ),
      layers.MaxPooling2D(2, padding="same"),
    ]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocs,
      layers.Conv2D(
        200, 3, activation="relu", padding="same", kernel_initializer="orthogonal"
      ),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(100, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(50, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(6, activation="softmax"),
    ]
  )


modele = construit_cnn_profond()
modele.summary()

In [ ]:
historique = entraine(modele, nb_epoques=15)

### Facultatif : normalisation par batch, AdamW et warmup

Cet exercice est facultatif, pour qui a fini en avance : la suite du TP n'en dépend pas. Il reprend le CNN profond avec trois techniques d'entraînement des réseaux profonds : la normalisation par batch, l'optimiseur AdamW, et un taux d'apprentissage qui monte pendant un *warmup* puis décroît en cosinus.

Écrivez `construit_cnn_profond_bn()`, qui renvoie le CNN profond de la solution avec une `BatchNormalization` dans chaque bloc, entre la convolution et son activation. Compilez le modèle qu'elle renvoie avec `keras.optimizers.AdamW`, dont le taux d'apprentissage suit un `keras.optimizers.schedules.CosineDecay` avec warmup, et entraînez-le 15 époques. Comparez enfin ses courbes d'apprentissage à celles du CNN profond (`historique`, de la solution précédente) : lequel dépasse le premier 70 % d'exactitude en validation ? Lequel finit le plus haut ? L'écart entre entraînement et validation change-t-il ?

**Documentation utile** :

- https://keras.io/api/layers/normalization_layers/batch_normalization/
- https://keras.io/api/optimizers/adamw/
- https://keras.io/api/optimizers/learning_rate_schedules/cosine_decay/

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une convolution de bloc, sans activation ni biais : `layers.Conv2D(200, 3, padding="same", kernel_initializer="orthogonal", use_bias=False)`, puis `layers.BatchNormalization()` et `layers.Activation("relu")`. La BatchNorm recentre les activations : un biais avant elle ne servirait à rien. La solution les renvoie d'une fonction, qui sert aux 6 blocs (chacun suivi de `layers.MaxPooling2D(2, padding="same")`) et à la 7ᵉ convolution. La tête ne change pas.
- Calendrier, compté en batchs : `pas = nb_epoques * len(dataset_ent)` (`len` donne le nombre de batchs d'une époque) et `warmup = pas // 20`, 5 % des pas. `keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.0, warmup_target=1e-3, warmup_steps=warmup, decay_steps=pas - warmup)` monte en ligne droite de 0 à 1e-3 pendant le warmup, puis redescend en cosinus jusqu'à 0. Ce pic vaut dix fois le 1e-4 de `entraine` : la BatchNorm tolère en général un taux d'apprentissage plus élevé. Il n'est pas réglé : un *learning rate range test* le choisirait.
- Optimiseur : `keras.optimizers.AdamW(calendrier, weight_decay=0.01)`, puis `optimiseur.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])` : ni les biais ni les paramètres des BatchNorm ne sont régularisés.
- `entraine` compile avec Adam à taux fixe : compilez vous-même, avec la même perte et la même métrique, puis `modele_bn.fit(dataset_ent, validation_data=dataset_val, epochs=nb_epoques)`, sans arrêt précoce : le calendrier prévoit toutes les époques, et le taux n'atteint 0 qu'à la dernière.
- Courbes : `trace_mesures(historique_bn)`, puis, pour comparer, `historique.history["val_accuracy"]` et `historique_bn.history["val_accuracy"]` sur un même graphique (deux `plt.plot(..., label=...)`), de même pour `"val_loss"`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_cnn_profond_bn() -> keras.Sequential:
  blocs = []
  for _ in range(6):
    blocs += [
      layers.Conv2D(...),
      layers.BatchNormalization(),
      ...,
      layers.MaxPooling2D(2, padding="same"),
    ]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocs,
      ...,
      layers.Flatten(),
      ...,
    ]
  )


modele_bn = construit_cnn_profond_bn()
nb_epoques = 15
pas = nb_epoques * len(dataset_ent)
warmup = pas // 20
calendrier = keras.optimizers.schedules.CosineDecay(...)
optimiseur = keras.optimizers.AdamW(calendrier, weight_decay=0.01)
optimiseur.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
modele_bn.compile(optimizer=optimiseur, loss=..., metrics=...)
historique_bn = modele_bn.fit(...)
trace_mesures(historique_bn)
for mesure, nom in (("val_accuracy", "Exactitude"), ("val_loss", "Perte")):
  plt.plot(historique.history[mesure], label="CNN profond : Adam à 1e-4")
  plt.plot(...)
  ...
  plt.show()
```

#### Solution

In [ ]:
def couches_conv_bn() -> list:
  """Une convolution sans biais ni activation, sa BatchNorm, puis la ReLU."""
  return [
    layers.Conv2D(
      200, 3, padding="same", kernel_initializer="orthogonal", use_bias=False
    ),
    layers.BatchNormalization(),  # la BatchNorm recentre : biais inutile
    layers.Activation("relu"),
  ]


def construit_cnn_profond_bn() -> keras.Sequential:
  """Le CNN profond de la solution, avec une BatchNorm dans chaque bloc."""
  blocs = []
  for _ in range(6):  # cartes de 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels de côté
    blocs += [*couches_conv_bn(), layers.MaxPooling2D(2, padding="same")]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocs,
      *couches_conv_bn(),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(100, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(50, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(6, activation="softmax"),
    ]
  )


modele_bn = construit_cnn_profond_bn()
modele_bn.summary()

In [ ]:
nb_epoques = 15
pas = nb_epoques * len(dataset_ent)  # en batchs
warmup = pas // 20  # 5 % des pas
calendrier = keras.optimizers.schedules.CosineDecay(
  initial_learning_rate=0.0,
  warmup_target=1e-3,  # pic, atteint à la fin du warmup
  warmup_steps=warmup,
  decay_steps=pas - warmup,
)
optimiseur = keras.optimizers.AdamW(calendrier, weight_decay=0.01)
# Ni les biais ni les paramètres des BatchNorm (gamma, beta) ne sont régularisés
optimiseur.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
modele_bn.compile(
  optimizer=optimiseur,
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
# Sans arrêt précoce : le calendrier prévoit toutes les époques
historique_bn = modele_bn.fit(
  dataset_ent, validation_data=dataset_val, epochs=nb_epoques
)
trace_mesures(historique_bn)

# Validation des deux CNN profonds, époque par époque
for mesure, nom in (("val_accuracy", "Exactitude"), ("val_loss", "Perte")):
  plt.plot(historique.history[mesure], label="CNN profond : Adam à 1e-4")
  plt.plot(historique_bn.history[mesure], label="BatchNorm, AdamW, warmup + cosinus")
  plt.title(f"{nom} de validation")
  plt.ylabel(nom)
  plt.xlabel("Époque")
  plt.legend()
  plt.show()

## Augmentation de données

Quand l'exactitude d'entraînement continue de monter alors que celle de validation stagne, le modèle surapprend. L'augmentation de données (slide « Augmenter les données par des couches ») lui présente à chaque époque des variantes des images d'entraînement. Ses couches n'agissent que pendant `fit` : `evaluate` et `predict` voient les images intactes.

Choisissez deux ou trois transformations qui gardent la classe d'un paysage, avec leur amplitude : lesquelles ne la gardent pas (un miroir vertical ? une rotation de 90° ? un zoom qui ne garde qu'un dixième de l'image ?) ? Placez leurs couches devant la même architecture, avec des poids neufs (`construit_cnn_profond()`), entraînez 15 époques, et comparez l'écart entre les courbes d'entraînement et de validation à celui du CNN profond sans augmentation.

**Documentation utile** :

- https://keras.io/api/layers/preprocessing_layers/image_augmentation/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Un modèle Keras s'emploie comme une couche : `keras.Sequential` accepte `construit_cnn_profond()` dans sa liste, après `keras.Input(shape=(150, 150, 3))` et les couches d'augmentation.
- Des couches qui gardent la classe d'un paysage, et leurs paramètres :
  - `layers.RandomFlip("horizontal")` : miroir gauche-droite, une image sur deux ;
  - `layers.RandomZoom((-0.3, 0))` : zoom avant jusqu'à 30 % (les valeurs négatives zooment en avant) ;
  - `layers.RandomRotation(0.03)` : rotation d'au plus 3 % d'un tour, environ 10°, dans un sens ou dans l'autre ;
  - `layers.RandomTranslation(0.1, 0.1)` : décalage d'au plus 10 % de la hauteur et de la largeur ;
  - `layers.RandomContrast(0.2)` : contraste multiplié par un facteur entre 0,8 et 1,2.
- À éviter : `layers.RandomFlip("vertical")` (le ciel passerait en bas), de grandes rotations, un zoom si fort qu'il ne garde qu'un bout de ciel ou de mur, un changement de teinte qui rendrait la mer verte ou un glacier jaune.
- Ces couches travaillent sur les pixels de 0 à 255, avant le `Rescaling` de `construit_cnn_profond()`.
- Entraînement : `entraine(modele, nb_epoques=15)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    ...,
    ...,
    construit_cnn_profond(),
  ]
)
modele.summary()
historique = entraine(modele, nb_epoques=15)
```

### Solution

In [ ]:
modele = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.RandomFlip("horizontal"),  # miroir, une image sur deux
    layers.RandomZoom((-0.3, 0)),  # zoom avant, jusqu'à 30 %
    construit_cnn_profond(),
  ]
)
modele.summary()
historique = entraine(modele, nb_epoques=15)

Une solution parmi d'autres : le miroir horizontal et le zoom avant. Chaque époque voit des variantes différentes des images : l'exactitude d'entraînement monte moins vite et l'écart avec la validation se réduit. Le modèle doit apprendre des caractéristiques qui résistent au cadrage et à l'orientation, au lieu de mémoriser les images d'entraînement. L'évaluation sur le test et l'analyse d'erreur portent sur ce modèle, `modele`.

## Architectures historiques

Trois architectures historiques, à entraîner puis comparer au CNN profond : LeNet et AlexNet, vues dans le chapitre sur les CNN, et un bloc Inception (GoogLeNet, 2014), présenté plus bas. Leur code est donné. Chacune a son propre nom de variable, et `modele` reste le CNN profond augmenté.

### LeNet

[LeNet-5](https://en.wikipedia.org/wiki/LeNet) (1998), conçu pour des chiffres manuscrits de 32 × 32 pixels : des convolutions 5 × 5, des activations sigmoïdes et des poolings par moyenne.

In [ ]:
le_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(6, 5, padding="same", activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Conv2D(16, 5, activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(120, activation="sigmoid"),
    layers.Dense(84, activation="sigmoid"),
    layers.Dense(6, activation="softmax"),
  ],
  name="le_net",
)
le_net.summary()
historique_le_net = entraine(le_net, nb_epoques=15)

### AlexNet

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) (2012), vainqueur d'ImageNet : une première convolution 11 × 11 de pas 4, des ReLU, du *dropout* dans une tête de deux couches de 4 096 neurones.

In [ ]:
alex_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(96, 11, strides=4, activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(256, 5, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Flatten(),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(6, activation="softmax"),
  ],
  name="alex_net",
)
alex_net.summary()
historique_alex_net = entraine(alex_net, nb_epoques=15)

### Bloc Inception

Un bloc [Inception](https://en.wikipedia.org/wiki/Inception_(deep_learning_architecture)) (GoogLeNet, 2014) : quatre branches en parallèle (convolutions 1 × 1, 3 × 3, 5 × 5 et pooling), concaténées sur l'axe des canaux. Un réseau complet en empile plusieurs. Comme dans GoogLeNet, le bloc vient ici après deux blocs convolution + pooling : ses convolutions 1 × 1 réduisent alors les 128 canaux reçus à 64, ce qui rend moins chères les convolutions 3 × 3 et 5 × 5 qui les suivent. Un pooling global termine le modèle, comme dans le CNN des slides : 249 606 paramètres en tout. Les branches parallèles demandent l'[API fonctionnelle de Keras](https://keras.io/guides/functional_api/), où chaque couche s'applique à un tenseur.

In [ ]:
def bloc_inception(entrees: keras.KerasTensor, filtres: int = 64) -> keras.KerasTensor:
  # conv 1 × 1
  branche_1 = layers.Conv2D(filtres, 1, activation="relu")(entrees)

  # conv 1 × 1 puis conv 3 × 3
  branche_3 = layers.Conv2D(filtres, 1, activation="relu")(entrees)
  branche_3 = layers.Conv2D(filtres, 3, padding="same", activation="relu")(branche_3)

  # conv 1 × 1 puis conv 5 × 5
  branche_5 = layers.Conv2D(filtres, 1, activation="relu")(entrees)
  branche_5 = layers.Conv2D(filtres, 5, padding="same", activation="relu")(branche_5)

  # max pooling 3 × 3 de pas 1 puis conv 1 × 1
  branche_pool = layers.MaxPooling2D(3, strides=1, padding="same")(entrees)
  branche_pool = layers.Conv2D(filtres, 1, activation="relu")(branche_pool)

  # concaténation sur l'axe des canaux : 4 × 64 = 256 canaux
  return layers.Concatenate()([branche_1, branche_3, branche_5, branche_pool])


entrees = keras.Input(shape=(150, 150, 3))
x = layers.Rescaling(1 / 255)(entrees)
# Deux blocs convolution + pooling : cartes de 36 × 36 à 128 canaux
x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = bloc_inception(x)  # convolutions 1 × 1 : de 128 à 64 canaux
x = layers.GlobalAveragePooling2D()(x)  # une valeur par canal
sorties = layers.Dense(6, activation="softmax")(x)
inception = keras.Model(entrees, sorties, name="inception")
inception.summary()
historique_inception = entraine(inception, nb_epoques=10)

## Évaluation des performances sur l'ensemble de test

Le dossier `seg_test` contient 3 000 images que le modèle n'a vues ni à l'entraînement, ni pour l'arrêt précoce : son exactitude sur elles estime honnêtement celle qu'il aura sur de nouvelles photos. `exactitudes_test` garde l'exactitude en test de chaque approche, pour les comparer à la fin du TP.

In [ ]:
perte, exactitude = modele.evaluate(dataset_test, verbose=0)
print(f"Exactitude en test : {exactitude:.3f}")
exactitudes_test = {"CNN profond, données augmentées": exactitude}

## Analyse d'erreur

On affiche la matrice de confusion, puis des images mal classées.

In [ ]:
def analyse_predictions(
  classes_predites: numpy.ndarray, etiquettes: numpy.ndarray
) -> None:
  matrice_confusion = sklearn.metrics.confusion_matrix(etiquettes, classes_predites)
  seaborn.heatmap(
    matrice_confusion,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=noms_etiquettes,
    yticklabels=noms_etiquettes,
  )
  plt.title("Matrice de confusion")
  plt.xlabel("Classe prédite")
  plt.ylabel("Classe réelle")
  plt.show()

  seaborn.countplot(x=[noms_etiquettes[prediction] for prediction in classes_predites])
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Classe")
  plt.show()


predictions_test = modele.predict(dataset_test, verbose=0).argmax(axis=-1)
analyse_predictions(predictions_test, etiquettes_test)

In [ ]:
def affiche_erreurs(classe_predite: str, classe_reelle: str) -> None:
  """Affiche jusqu'à 25 images de test de classe_reelle prédites classe_predite."""
  print(f"Prédiction : {classe_predite}, classe réelle : {classe_reelle}")
  erreurs = chemins_test[
    (predictions_test == noms_etiquettes.index(classe_predite))
    & (etiquettes_test == noms_etiquettes.index(classe_reelle))
  ]
  erreurs = numpy.random.choice(erreurs, size=min(len(erreurs), 25), replace=False)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  indices_grille = list(itertools.product(range(5), repeat=2))
  for i, j in indices_grille:
    ax[i, j].axis("off")
  for chemin, (i, j) in zip(erreurs, indices_grille):
    ax[i, j].imshow(keras.utils.load_img(chemin))
  plt.show()

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées montagne
affiche_erreurs("glacier", "mountain")

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées mer
affiche_erreurs("glacier", "sea")

In [ ]:
# Images prédites bâtiments alors qu'elles sont étiquetées rue
affiche_erreurs("buildings", "street")

## Transfert d'apprentissage

On réutilise un réseau pré-entraîné sur ImageNet, EfficientNetV2B0 (`keras.applications`), comme extracteur de caractéristiques : ses poids sont gelés, et on n'entraîne qu'une nouvelle tête de classification pour nos 6 classes (slides du chapitre « Transfer learning »).

Les poids pré-entraînés attendent les images telles qu'à leur entraînement :

- **Taille** : 224 × 224, la taille des images d'ImageNet sur lesquelles les poids ont appris. On recharge les paysages à cette taille.
- **Pixels de 0 à 255** : EfficientNetV2 les normalise lui-même, dans sa première couche. Le modèle ne commence donc pas par `Rescaling(1 / 255)`, contrairement aux CNN entraînés depuis zéro : il ne verrait que des images presque noires.

`modele_base.trainable = False` exclut les poids d'EfficientNetV2 de la descente de gradient. Ses couches `BatchNormalization` deviennent alors non entraînables : elles normalisent avec les statistiques d'ImageNet, sans les mettre à jour sur nos batchs de paysages.

In [ ]:
TAILLE_TRANSFERT = (224, 224)  # taille d'entraînement des poids ImageNet
# Mêmes images, dans le même ordre : chemins_test et etiquettes_test restent valables
dataset_ent, dataset_val, dataset_test = charge_paysages(TAILLE_TRANSFERT)

In [ ]:
modele_base = keras.applications.EfficientNetV2B0(
  include_top=False,  # sans la tête ImageNet (1000 classes)
  weights="imagenet",
  pooling="avg",  # GlobalAveragePooling2D : un vecteur de 1280 valeurs par image
)
modele_base.trainable = False  # poids figés, BatchNorm en inférence

entrees = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = modele_base(entrees)
sorties = layers.Dense(6, activation="softmax")(x)  # 6 paysages
modele = keras.Model(entrees, sorties)
modele.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
modele.summary()

In [ ]:
historique = modele.fit(
  dataset_ent, validation_data=dataset_val, epochs=10, callbacks=[cree_arret_precoce()]
)
trace_mesures(historique)

In [ ]:
perte, exactitude = modele.evaluate(dataset_test, verbose=0)
print(f"Exactitude en test : {exactitude:.3f}")
exactitudes_test["EfficientNetV2B0 gelé"] = exactitude

predictions_test = modele.predict(dataset_test, verbose=0).argmax(axis=-1)
analyse_predictions(predictions_test, etiquettes_test)
affiche_erreurs("glacier", "mountain")
affiche_erreurs("glacier", "sea")
affiche_erreurs("buildings", "street")

### Questions

1. Combien de paramètres ce modèle entraîne-t-il, et combien en gèle-t-il ? (voir son résumé)
2. Comparez son exactitude de test à celle du CNN profond entraîné depuis zéro, et le temps de leurs entraînements.
3. Les erreurs restantes (matrice de confusion, images mal classées) ressemblent-elles à celles du CNN ?

#### Solution

1. Seule la tête est entraînée : 1 280 × 6 + 6 = 7 686 paramètres, contre 5 919 312 gelés dans EfficientNetV2B0.
2. Le modèle pré-entraîné dépasse le CNN profond, 93 % contre 86 % environ en test, dès les premières époques et en quelques secondes par époque : seule la tête apprend. Les caractéristiques qu'EfficientNetV2 a apprises sur plus d'un million d'images d'ImageNet se transfèrent bien à nos 14 000 paysages.
3. Les confusions qui restent opposent surtout des classes réellement proches (glacier et montagne, bâtiments et rue) ; beaucoup de ces images sont ambiguës, voire mal annotées.

### Fine-tuning : dégeler le backbone

Une fois la tête entraînée, on peut dégeler EfficientNetV2 et entraîner tout le réseau quelques époques de plus, avec un taux d'apprentissage 100 fois plus faible : ses poids sont déjà bons, on veut seulement les ajuster à nos paysages.

La cellule suivante garde d'abord les poids du modèle à tête entraînée (`modele.save_weights`), pour repartir du même point au dégel partiel, puis fait le fine-tuning des slides (« Fine-tuning : dégeler le backbone ») : `modele_base` dégelé en entier, sauf ses BatchNorm, qui gardent les statistiques d'ImageNet, et Adam à 1e-5 pendant 3 époques. Keras applique un seul taux d'apprentissage à tout le modèle : la tête, déjà entraînée, avance elle aussi à 1e-5.

In [ ]:
# Poids d'après l'entraînement de la tête, pour en repartir au dégel partiel
modele.save_weights("tete.weights.h5")

modele_base.trainable = True
# BatchNorm gelées : statistiques d'ImageNet conservées
for couche in modele_base.layers:
  if isinstance(couche, layers.BatchNormalization):
    couche.trainable = False

# À refaire après tout changement de trainable
modele.compile(
  optimizer=keras.optimizers.Adam(1e-5),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
modele.summary()
historique = modele.fit(dataset_ent, validation_data=dataset_val, epochs=3)
trace_mesures(historique)
perte, exactitude = modele.evaluate(dataset_test, verbose=0)
print(f"Exactitude en test : {exactitude:.3f}")
exactitudes_test["EfficientNetV2B0, dégel complet"] = exactitude

L'exactitude gagne encore un point environ, de 93 à 94 % en test : le backbone s'adapte à nos paysages. Un taux d'apprentissage trop élevé détruirait au contraire ce qu'il a appris sur ImageNet.

### Dégel partiel

Avec peu de données, on ne dégèle souvent que les dernières couches du backbone (la slide « Fine-tuning : dégeler le backbone » le suggère en commentaire) : les premières, qui détectent des bords et des textures, servent telles quelles d'un jeu d'images à l'autre.

Repartez des poids d'après l'entraînement de la tête, puis dégelez seulement les 40 dernières couches de `modele_base`, BatchNorm toujours gelées. Recompilez avec Adam à 1e-5, entraînez 3 époques et ajoutez l'exactitude en test à `exactitudes_test`. Comparez au dégel complet : nombre de paramètres entraînables, durée d'une époque, exactitude de validation et de test.

**Documentation utile** :

- https://keras.io/api/models/model_saving_apis/weights_saving_and_loading/

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `modele.load_weights("tete.weights.h5")` rend au modèle les poids d'avant le dégel complet, tête comprise.
- `modele_base.trainable = True` ne descend à ses couches que s'il change la valeur : après le dégel complet, il vaut déjà `True` et ne ferait rien. Réglez donc chaque couche : `trainable = False` sur `modele_base.layers[:-40]`, toutes sauf les 40 dernières (la fin du dernier bloc, `block6f`, et la convolution finale), et, sur `modele_base.layers[-40:]`, `True` sauf pour les `layers.BatchNormalization`.
- Les 40 dernières couches commencent à `block6f_se_expand`, au milieu du bloc `block6f`. L'exemple de keras.io *Image classification via fine-tuning with EfficientNet* dégèle plutôt des blocs entiers, à cause de la connexion résiduelle de l'entrée à la sortie de chaque bloc : variante à essayer, le test `couche.name.startswith(("block6", "top"))`.
- `modele.compile(...)`, comme au dégel complet : sans lui, le changement de `trainable` est ignoré. `modele.summary()` affiche ensuite le nombre de paramètres entraînables.
- `modele.fit(dataset_ent, validation_data=dataset_val, epochs=3)`, puis `trace_mesures` et `modele.evaluate(dataset_test, verbose=0)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele.load_weights(...)

for couche in modele_base.layers[...]:
  couche.trainable = ...
for couche in modele_base.layers[...]:
  couche.trainable = not isinstance(couche, ...)

modele.compile(...)
modele.summary()
historique = modele.fit(dataset_ent, validation_data=dataset_val, epochs=...)
trace_mesures(historique)
perte, exactitude = modele.evaluate(dataset_test, verbose=0)
print(f"Exactitude en test : {exactitude:.3f}")
exactitudes_test["EfficientNetV2B0, dégel partiel"] = exactitude
```

#### Solution

In [ ]:
modele.load_weights("tete.weights.h5")  # la tête entraînée, le backbone d'ImageNet

# modele_base.trainable vaut déjà True : on règle chaque couche
for couche in modele_base.layers[:-40]:  # toutes les couches sauf les 40 dernières
  couche.trainable = False
for couche in modele_base.layers[-40:]:  # les 40 dernières, BatchNorm gelées
  couche.trainable = not isinstance(couche, layers.BatchNormalization)

modele.compile(
  optimizer=keras.optimizers.Adam(1e-5),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
modele.summary()
historique = modele.fit(dataset_ent, validation_data=dataset_val, epochs=3)
trace_mesures(historique)
perte, exactitude = modele.evaluate(dataset_test, verbose=0)
print(f"Exactitude en test : {exactitude:.3f}")
exactitudes_test["EfficientNetV2B0, dégel partiel"] = exactitude

- Le dégel partiel entraîne 1 660 134 paramètres (1 652 448 dans les 40 dernières couches de `modele_base`, 7 686 dans la tête), contre 5 805 782 pour le dégel complet. Le gradient ne traverse que la fin du réseau : chaque époque est plus courte.
- Le dégel partiel peut suffire ici, ce que vos exactitudes de test diront : nos photos sont proches de celles d'ImageNet, et les premières couches d'EfficientNetV2 y ont peu à réapprendre. Sur des images qui s'en éloignent (radiographies, vues satellite), le dégel complet a plus à gagner.
- La suite du TP utilise `modele`, fine-tuné en dégel partiel.

## Prédire dans des conditions « réelles »

Le dossier `seg_pred` contient des images sans étiquette : on ne peut pas y mesurer l'exactitude. On peut en revanche afficher des photos et les probabilités que le modèle attribue à chaque classe.

In [ ]:
chemins_pred = sorted(pathlib.Path("landscape/seg_pred").rglob("*.jpg"))
print(f"{len(chemins_pred)} images sans étiquette")

In [ ]:
# 25 images au hasard, sans remise, chargées à la taille du modèle
indices = numpy.random.choice(len(chemins_pred), size=25, replace=False)
images_pred = numpy.stack(
  [
    keras.utils.img_to_array(
      keras.utils.load_img(chemins_pred[indice], target_size=TAILLE_TRANSFERT)
    )
    for indice in indices
  ]
)
probabilites = modele.predict(images_pred, verbose=0)

# Chaque image, puis la distribution de ses prédictions sur la ligne d'en dessous
_, ax = plt.subplots(10, 5, figsize=(30, 45))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  classe_predite = noms_etiquettes[probabilites[k].argmax()]
  ax[i * 2, j].imshow(images_pred[k].astype("uint8"))
  ax[i * 2, j].set_title(f"{chemins_pred[indices[k]].name} ({classe_predite})")
  ax[i * 2, j].axis("off")
  ax[i * 2 + 1, j].bar(noms_etiquettes, probabilites[k])
plt.show()

## Exactitude en fonction de la probabilité maximale

L'exactitude varie-t-elle selon la probabilité maximale que rend le modèle ? Si oui, on peut ne garder que les prédictions sûres, et confier les autres à un humain.

In [ ]:
probabilites_test = modele.predict(dataset_test, verbose=0)

In [ ]:
def calcule_exactitude(
  probabilites: numpy.ndarray, etiquettes: numpy.ndarray, seuil: float
) -> tuple[float, float, float, float]:
  predictions = probabilites.argmax(axis=-1)
  masque_haut = probabilites.max(axis=-1) > seuil
  masque_bas = ~masque_haut

  nb_haut = masque_haut.sum()
  nb_bas = masque_bas.sum()

  if nb_haut:
    haut = (predictions[masque_haut] == etiquettes[masque_haut]).sum() / nb_haut
  else:
    haut = 1.0
  if nb_bas:
    bas = (predictions[masque_bas] == etiquettes[masque_bas]).sum() / nb_bas
  else:
    bas = 1.0
  return haut, bas, nb_haut / etiquettes.shape[0], nb_bas / etiquettes.shape[0]


exactitude_haut, exactitude_bas, part_haut, part_bas = calcule_exactitude(
  probabilites_test, etiquettes_test, 0.99
)
print(
  "Exactitude pour les prédictions du modèle qui ont une probabilité au "
  f"dessus du seuil ({part_haut * 100:.2f}% des données) : "
  f"{exactitude_haut:.2f}"
)
print(
  "Exactitude pour les prédictions du modèle qui ont une probabilité en "
  f"dessous du seuil ({part_bas * 100:.2f}% des données) : "
  f"{exactitude_bas:.2f}"
)

xs = numpy.arange(101) / 100
exactitudes = []
parts_haut = []
for x in xs:
  exactitude, _, part_haut, _ = calcule_exactitude(
    probabilites_test, etiquettes_test, x
  )
  exactitudes.append(exactitude)
  parts_haut.append(part_haut)
plt.plot(xs, exactitudes, label="Exactitude")
plt.plot(xs, parts_haut, label="Proportion de prédictions retenues")
plt.xlabel("Seuil de prédiction")
plt.title("Exactitude et proportion de prédictions retenues selon le seuil")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

Partie facultative : les slides présentent le *Vision Transformer* plus tard (chapitre « Transformers ») ; ce qu'il faut en savoir ici tient dans le paragraphe suivant.

Même démarche avec un *Vision Transformer* (ViT) pré-entraîné sur ImageNet-21k (14 millions d'images), chargé avec KerasHub. Le ViT découpe une image 224 × 224 en 196 carrés de 16 × 16 pixels, et ajoute devant eux un token de classification, `[CLS]`. Le dernier état de ce token résume l'image : la tête de classification se branche sur lui.

Construisez `modele_vit` :

- le préréglage `"vit_base_patch16_224_imagenet21k"`, chargé deux fois : `keras_hub.layers.ViTImageConverter`, ses prétraitements (redimensionner en 224 × 224, ramener les pixels de 0–255 à [-1, 1]), et `keras_hub.models.ViTBackbone`, le réseau, à geler ;
- en API fonctionnelle : entrée 224 × 224 × 3, prétraitements, ViT, token `[CLS]`, puis une couche dense de sortie à 6 classes ;
- compilé comme le modèle EfficientNetV2.

**Documentation utile** :

- https://keras.io/keras_hub/api/models/vit/vit_backbone/
- https://keras.io/keras_hub/api/models/vit/vit_image_converter/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Chargement : `keras_hub.layers.ViTImageConverter.from_preset(PREREGLAGE)` et `keras_hub.models.ViTBackbone.from_preset(PREREGLAGE)` (330 Mo à télécharger).
- Le convertisseur s'emploie comme une couche, juste après `keras.Input(shape=(224, 224, 3))`.
- Le ViT renvoie un vecteur de 768 valeurs par token, de forme `(None, 197, 768)` : le token `[CLS]` est le premier, `x[:, 0]`.
- `vit.trainable = False` suffit : le ViT normalise avec des `LayerNormalization`, qui n'ont pas de statistiques à figer, contrairement aux BatchNorm.
- Compilation : `keras.optimizers.Adam(1e-3)`, `loss="sparse_categorical_crossentropy"`, `metrics=["accuracy"]`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
PREREGLAGE = "vit_base_patch16_224_imagenet21k"
convertisseur = keras_hub.layers.ViTImageConverter.from_preset(PREREGLAGE)
vit = ...
vit.trainable = ...

entrees = keras.Input(shape=(224, 224, 3))
x = convertisseur(entrees)
x = ...
x = x[...]
sorties = ...
modele_vit = keras.Model(entrees, sorties, name="vit_model")
modele_vit.compile(...)
modele_vit.summary()
```

### Solution

In [ ]:
PREREGLAGE = "vit_base_patch16_224_imagenet21k"
convertisseur = keras_hub.layers.ViTImageConverter.from_preset(PREREGLAGE)
vit = keras_hub.models.ViTBackbone.from_preset(PREREGLAGE)
vit.trainable = False

entrees = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = convertisseur(entrees)  # 224 × 224, pixels dans [-1, 1]
x = vit(x)  # (None, 197, 768) : un vecteur par token
x = x[:, 0]  # le token [CLS], qui résume l'image
sorties = layers.Dense(6, activation="softmax")(x)
modele_vit = keras.Model(entrees, sorties, name="vit_model")
modele_vit.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
modele_vit.summary()

### Comparaison des trois approches

Entraînez `modele_vit` 5 époques (environ une minute chacune sur le GPU de Colab), évaluez-le sur l'ensemble de test, puis comparez les trois approches : CNN entraîné depuis zéro, EfficientNetV2B0 pré-entraîné, ViT pré-entraîné. Qu'apporte le transfert d'apprentissage, et à quel prix ?

In [ ]:
historique = modele_vit.fit(
  dataset_ent, validation_data=dataset_val, epochs=5, callbacks=[cree_arret_precoce()]
)
trace_mesures(historique)

In [ ]:
perte, exactitude = modele_vit.evaluate(dataset_test, verbose=0)
exactitudes_test["ViT gelé"] = exactitude

predictions_test = modele_vit.predict(dataset_test, verbose=0).argmax(axis=-1)
analyse_predictions(predictions_test, etiquettes_test)

for nom, exactitude in exactitudes_test.items():
  print(f"{nom} : {exactitude:.3f}")

#### Solution

- Les deux modèles pré-entraînés dépassent nettement le CNN entraîné depuis zéro (93 à 94 % contre 86 % environ), pour une fraction de son temps d'entraînement : c'est tout l'intérêt du transfert d'apprentissage quand on a peu de données (14 000 images ici, contre plus d'un million pour ImageNet et 14 millions pour ImageNet-21k).
- Entre EfficientNetV2B0 et le ViT, l'écart est faible, mais le ViT est bien plus gros : 86 millions de paramètres contre 5,9 millions, et chaque prédiction coûte en proportion. Le fine-tuning du backbone apporte encore un point.
- Les confusions restantes sont les mêmes pour les trois modèles : glacier et montagne, bâtiments et rue.